# IDRiD — **Multiclasse (0–4)** DR — *Boosted*
EfficientNet‑B3 @ 380 + K‑Means (pseudo‑labels com Hungarian) + Pré‑treino da cabeça + **Fine‑tuning supervisionado** com **Class‑Balanced Focal Loss**, **EMA**, **Mixup**, **TTA** e **OneCycleLR (40 epochs)**

In [9]:
# Determinismo do CuBLAS deve ser configurado **antes** de importar torch
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"  # ou ":4096:8"
print("CUBLAS_WORKSPACE_CONFIG =", os.environ.get("CUBLAS_WORKSPACE_CONFIG"))

CUBLAS_WORKSPACE_CONFIG = :16:8


In [10]:
# --- Imports
import os, math, time, copy, random, json
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import label_binarize
from scipy.optimize import linear_sum_assignment

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

import warnings; warnings.filterwarnings("ignore")
print("Torch:", torch.__version__)

Torch: 2.6.0+cu124


In [11]:
# --- Configs
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Tamanho e modelo
IMG_SIZE = 380  # B3 input
BACKBONE_WEIGHTS = EfficientNet_B3_Weights.IMAGENET1K_V1

# Treino
BATCH_SIZE = 24           # 380px é pesado; ajuste se necessário
NUM_WORKERS = 2
EPOCHS = 40               # OneCycleLR
WARM_FREEZE_EPOCHS = 3    # congela backbone nos primeiros epochs

# Otimizador
BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4

# Mixup/TTA
MIXUP_ALPHA = 0.2
USE_TTA = True

# Focal + Class-Balanced
CB_BETA = 0.999  # 0.99–0.9999
FOCAL_GAMMA = 2.0

# Paths (ajuste para Kaggle/Local)
# Kaggle (exemplo):
CSV_PATH = "/kaggle/input/idrid-dataset/idrid_labels.csv"
IMG_DIR  = "/kaggle/input/idrid-dataset/Imagenes/Imagenes"
# Local (default):
# CSV_PATH = "./idrid/idrid_labels.csv"
# IMG_DIR  = "./idrid/Imagenes/Imagenes"

OUT_DIR = Path("./outputs"); OUT_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

seed_everything(SEED)

In [13]:
# --- Preprocess helpers
def circular_crop(img: np.ndarray, ratio: float = 0.9):
    h, w = img.shape[:2]
    center = (w//2, h//2)
    radius = int(min(center) * ratio)
    Y, X = np.ogrid[:h, :w]
    dist = (X - center[0])**2 + (Y - center[1])**2
    mask = dist <= radius**2
    out = np.zeros_like(img)
    out[mask] = img[mask]
    return out

def clahe_green(img: np.ndarray, clip=2.0, grid=8):
    if img.ndim == 2:
        g = img
    else:
        g = img[:,:,1]
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
    g_eq = clahe.apply(g)
    if img.ndim == 2:
        return g_eq
    out = img.copy()
    out[:,:,1] = g_eq
    return out

def build_img_path(img_dir: str, img_id: str):
    for ext in (".jpg",".jpeg",".png",".tif",".bmp",".JPG",".PNG",".TIFF"):
        p = Path(img_dir) / f"{img_id}{ext}"
        if p.exists():
            return str(p)
    return str(Path(img_dir) / f"{img_id}.jpg")

In [14]:
# --- Dataset
class IDRiDDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: str, transform=None, use_circ_crop=False, use_clahe=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.use_circ_crop = use_circ_crop
        self.use_clahe = use_clahe

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["id_code"]
        y = int(row["diagnosis"])  # 0..4
        p = build_img_path(self.img_dir, img_id)
        img = cv2.imread(p, cv2.IMREAD_COLOR)
        if img is None: raise FileNotFoundError(f"Imagem não encontrada: {p}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.use_circ_crop: img = circular_crop(img)
        if self.use_clahe:     img = clahe_green(img)
        img_pil = Image.fromarray(img)
        x = self.transform(img_pil) if self.transform else transforms.ToTensor()(img_pil)
        return x, y, img_id

In [15]:
# --- Transforms @380 com jitter suave
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.RandomCrop(IMG_SIZE, pad_if_needed=True),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(7),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [16]:
# --- Split estratificado 70/15/15 + DataLoaders determinísticos
df = pd.read_csv(CSV_PATH)
assert {"id_code","diagnosis"}.issubset(df.columns)
df["diagnosis"] = df["diagnosis"].astype(int)

from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["diagnosis"], random_state=SEED)
val_df, test_df  = train_test_split(temp_df, test_size=0.50, stratify=temp_df["diagnosis"], random_state=SEED)

NUM_CLASSES = int(df["diagnosis"].nunique())
print("Classes:", sorted(df["diagnosis"].unique()), "→ NUM_CLASSES =", NUM_CLASSES)

train_ds = IDRiDDataset(train_df, IMG_DIR, transform=train_tfms, use_circ_crop=False, use_clahe=True)
val_ds   = IDRiDDataset(val_df,   IMG_DIR, transform=val_tfms,   use_circ_crop=False, use_clahe=True)
test_ds  = IDRiDDataset(test_df,  IMG_DIR, transform=val_tfms,   use_circ_crop=False, use_clahe=True)

# Sampler balanceado por classe
labels = train_df["diagnosis"].to_numpy()
class_counts = np.bincount(labels, minlength=NUM_CLASSES)
class_weights_sampler = 1.0 / (class_counts + 1e-6)
sample_weights = class_weights_sampler[labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# Semeadura por worker + generator
def seed_worker(worker_id):
    import torch, numpy as np, random
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)
g = torch.Generator(); g.manual_seed(SEED)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker, generator=g)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker, generator=g)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, worker_init_fn=seed_worker, generator=g)

print("Tamanhos:", len(train_ds), len(val_ds), len(test_ds))

Classes: [0, 1, 2, 3, 4] → NUM_CLASSES = 5
Tamanhos: 318 68 69


In [17]:
# --- Modelo: EfficientNet-B3
backbone = efficientnet_b3(weights=BACKBONE_WEIGHTS)
in_features = backbone.classifier[1].in_features

# Extrator de embeddings
embed_net = copy.deepcopy(backbone)
embed_net.classifier = nn.Identity()
embed_net = embed_net.to(DEVICE).eval()

def make_head(out_dim: int, dropout: float = 0.3):
    return nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, out_dim))

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth
100%|██████████| 47.2M/47.2M [00:00<00:00, 179MB/s]


In [18]:
@torch.inference_mode()
def extract_embeddings(dloader, net):
    feats, ys, ids = [], [], []
    for xb, yb, idb in dloader:
        xb = xb.to(DEVICE, non_blocking=True)
        fb = net(xb).detach().cpu().numpy()
        feats.append(fb); ys.append(yb.numpy()); ids.extend(list(idb))
    feats = np.concatenate(feats, axis=0)
    ys = np.concatenate(ys, axis=0)
    return feats, ys, ids

print("Extraindo embeddings...")
train_feats, train_y, train_ids = extract_embeddings(train_loader, embed_net)
val_feats,   val_y,   val_ids   = extract_embeddings(val_loader,   embed_net)
test_feats,  test_y,  test_ids  = extract_embeddings(test_loader,  embed_net)

def l2norm(x):
    n = np.linalg.norm(x, axis=1, keepdims=True) + 1e-9
    return x / n

train_feats = l2norm(train_feats); val_feats = l2norm(val_feats); test_feats = l2norm(test_feats)
print("Embeddings:", train_feats.shape)

Extraindo embeddings...
Embeddings: (318, 1536)


In [19]:
# --- K-Means + Hungarian mapping cluster→classe
k = NUM_CLASSES
kmeans = KMeans(n_clusters=k, n_init=10, random_state=SEED)
kmeans.fit(train_feats)
train_clusters = kmeans.predict(train_feats)
val_clusters   = kmeans.predict(val_feats)
test_clusters  = kmeans.predict(test_feats)

# Confusão cluster vs label em VAL → Hungarian
C = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for c, y in zip(val_clusters, val_y):
    C[c, y] += 1
row_ind, col_ind = linear_sum_assignment(C.max() - C)
cluster_to_label = {row: col for row, col in zip(row_ind, col_ind)}
print("Mapeamento cluster→classe:", cluster_to_label)

# Pseudo‑rótulos mapeados
train_pseudo = np.array([cluster_to_label[c] for c in train_clusters])
val_pseudo   = np.array([cluster_to_label[c] for c in val_clusters])

# (Opcional) filtrar baixa confiança pela distância ao centróide
def confidence_mask(feats, clusters, centers, q=0.7):
    d = np.linalg.norm(feats - centers[clusters], axis=1)
    thr = np.quantile(d, q)  # mantém os mais próximos (<= thr)
    return d <= thr

keep_train = confidence_mask(train_feats, train_clusters, kmeans.cluster_centers_, q=0.7)
keep_val   = confidence_mask(val_feats,   val_clusters,   kmeans.cluster_centers_, q=0.7)
print(f"Pseudo train kept: {keep_train.mean():.2%}, val kept: {keep_val.mean():.2%}")

Mapeamento cluster→classe: {0: 1, 1: 4, 2: 0, 3: 3, 4: 2}
Pseudo train kept: 69.81%, val kept: 69.12%


In [20]:
# --- Datasets com pseudo‑rótulos
class PseudoLabelDataset(Dataset):
    def __init__(self, base_ds: IDRiDDataset, pseudo_labels: np.ndarray, mask=None):
        self.base = base_ds
        if mask is None: mask = np.ones(len(base_ds), dtype=bool)
        self.indices = np.nonzero(mask)[0]
        self.pseudo = pseudo_labels[self.indices].astype(int)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = int(self.indices[i])
        x, _, img_id = self.base[idx]
        return x, int(self.pseudo[i]), img_id

train_pl_ds = PseudoLabelDataset(train_ds, train_pseudo, keep_train)
val_pl_ds   = PseudoLabelDataset(val_ds,   val_pseudo,   keep_val)

train_pl_loader = DataLoader(train_pl_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_pl_loader   = DataLoader(val_pl_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print("Pseudo sizes:", len(train_pl_ds), len(val_pl_ds))

Pseudo sizes: 222 47


In [21]:
# --- Losses, EMA, Mixup, Metrics, TTA
def effective_num_weights(counts, beta=0.999, K=None):
    eff_num = 1.0 - np.power(beta, counts)
    w = (1.0 - beta) / (eff_num + 1e-12)
    if K is None: K = len(counts)
    w = w / w.sum() * K
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    def forward(self, logits, target):
        logp = torch.log_softmax(logits, dim=1)
        p = torch.exp(logp)
        pt = p.gather(1, target.unsqueeze(1)).squeeze(1)
        focal = (1 - pt).pow(self.gamma) * (-logp.gather(1, target.unsqueeze(1)).squeeze(1))
        if self.weight is not None: focal = focal * self.weight[target]
        return focal.mean()

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = [p.detach().clone() for p in model.parameters() if p.requires_grad]
        self.params = [p for p in model.parameters() if p.requires_grad]
        self.backup = None
    @torch.no_grad()
    def update(self):
        for s, p in zip(self.shadow, self.params):
            s.mul_(self.decay).add_(p, alpha=1 - self.decay)
    @torch.no_grad()
    def store(self):
        self.backup = [p.detach().clone() for p in self.params]
    @torch.no_grad()
    def copy_to(self):
        for p, s in zip(self.params, self.shadow): p.copy_(s)
    @torch.no_grad()
    def restore(self):
        for p, b in zip(self.params, self.backup): p.copy_(b)

def mixup_data(x, y, alpha=0.2):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    bs = x.size(0)
    index = torch.randperm(bs, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b,)

def eval_metrics_from_logits(logits_t, y_true_t, num_classes):
    if logits_t.numel() == 0:
        return {"acc":0.0,"macro_f1":0.0,"roc_auc_ovr":0.0,"pr_auc_macro":0.0}
    y_true = y_true_t.numpy()
    y_prob = torch.softmax(logits_t, dim=1).numpy()
    y_pred = y_prob.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    try:
        y_bin = label_binarize(y_true, classes=list(range(num_classes)))
        roc_auc = roc_auc_score(y_bin, y_prob, multi_class="ovr", average="macro")
    except Exception:
        roc_auc = float("nan")
    try:
        pr_auc = average_precision_score(y_bin, y_prob, average="macro")
    except Exception:
        pr_auc = float("nan")
    return {"acc":acc, "macro_f1":macro_f1, "roc_auc_ovr":roc_auc, "pr_auc_macro":pr_auc}

@torch.inference_mode()
def predict_tta(model, xb):
    outs = [model(xb)]
    outs.append(model(torch.flip(xb, dims=[-1])))
    outs.append(model(torch.flip(xb, dims=[-2])))
    return torch.stack(outs, 0).softmax(2).mean(0)

In [22]:
# --- Pré‑treino da cabeça com pseudo‑rótulos (congela embed)
for p in embed_net.parameters(): p.requires_grad = False
head_pre = make_head(NUM_CLASSES, dropout=0.3)
model_pre = nn.Sequential(embed_net, head_pre).to(DEVICE)

criterion_pre = nn.CrossEntropyLoss()
optimizer_pre = torch.optim.AdamW(model_pre.parameters(), lr=1e-3, weight_decay=1e-5)

print("Pré‑treino (pseudo‑labels mapeados + confiança)...")
for epoch in range(1, 5):
    model_pre.train()
    losses = []
    for xb, yb, _ in train_pl_loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE)
        optimizer_pre.zero_grad(set_to_none=True)
        logits = model_pre(xb)
        loss = criterion_pre(logits, yb)
        loss.backward(); optimizer_pre.step()
        losses.append(loss.item())
    model_pre.eval()
    with torch.inference_mode():
        val_losses = []
        for xb, yb, _ in val_pl_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            val_losses.append(criterion_pre(model_pre(xb), yb).item())
    print(f"[PRE] ep={epoch:02d} | Ltr={np.mean(losses):.4f} Lva={np.mean(val_losses):.4f}")

Pré‑treino (pseudo‑labels mapeados + confiança)...
[PRE] ep=01 | Ltr=1.6119 Lva=1.5772
[PRE] ep=02 | Ltr=1.5744 Lva=1.6009
[PRE] ep=03 | Ltr=1.5489 Lva=1.5454
[PRE] ep=04 | Ltr=1.5265 Lva=1.5376


In [23]:
# --- Fine‑tuning supervisionado com CB‑Focal, EMA, Mixup e OneCycleLR
def build_supervised_model(dropout=0.3):
    head_sup = make_head(NUM_CLASSES, dropout=dropout)
    model = nn.Sequential(embed_net, head_sup).to(DEVICE)
    return model

cb_weights_t = effective_num_weights(class_counts, beta=CB_BETA, K=NUM_CLASSES)
criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=cb_weights_t)

model = build_supervised_model(dropout=0.3)

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=BASE_LR, steps_per_epoch=len(train_loader), epochs=EPOCHS,
    pct_start=0.1, div_factor=25, final_div_factor=1e4
)

ema = EMA(model, decay=0.999)

def set_backbone_trainable(require_grad: bool):
    for p in embed_net.parameters(): p.requires_grad = require_grad

set_backbone_trainable(False)
best = {"val_loss": float("inf"), "metrics": None, "state": None}

for epoch in range(1, EPOCHS+1):
    if epoch == WARM_FREEZE_EPOCHS + 1:
        set_backbone_trainable(True)

    model.train()
    losses = []
    for xb, yb, _ in train_loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        if MIXUP_ALPHA > 0:
            xb, y_a, y_b, lam = mixup_data(xb, yb, alpha=MIXUP_ALPHA)
            logits = model(xb)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
        else:
            logits = model(xb)
            loss = criterion(logits, yb)
        loss.backward()
        optimizer.step(); sched.step()
        ema.update()
        losses.append(loss.item())

    # Val with EMA weights
    model.eval(); ema.store(); ema.copy_to()
    with torch.inference_mode():
        val_losses = []; logits_all = []; y_all = []
        for xb, yb, _ in val_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            logits = model(xb) if not USE_TTA else predict_tta(model, xb)
            val_losses.append(nn.CrossEntropyLoss()(logits, yb).item())
            logits_all.append(logits.cpu()); y_all.append(yb.cpu())
        logits_all = torch.cat(logits_all, 0); y_all = torch.cat(y_all, 0)
        met = eval_metrics_from_logits(logits_all, y_all, NUM_CLASSES)
        val_loss = float(np.mean(val_losses))
        if val_loss < best["val_loss"]:
            best.update(val_loss=val_loss, metrics=met, state=copy.deepcopy(model.state_dict()))
            torch.save(model.state_dict(), OUT_DIR / "best_model_multiclass_boosted.pth")
            json.dump({"val_loss": val_loss, **met, "lr": BASE_LR, "wd": WEIGHT_DECAY, "epochs": EPOCHS},
                      open(OUT_DIR / "best_hyperparams_multiclass_boosted.json","w"), indent=2)
    ema.restore()
    print(f"[SUP] ep={epoch:02d} | Ltr={np.mean(losses):.4f} Lva={val_loss:.4f} | acc={met['acc']:.3f} F1={met['macro_f1']:.3f}")

# Carrega melhor estado
model.load_state_dict(best["state"])
torch.save(model.state_dict(), OUT_DIR / "best_model_multiclass_boosted.pth")
print("Modelo salvo em:", OUT_DIR / "best_model_multiclass_boosted.pth")

[SUP] ep=01 | Ltr=0.9998 Lva=1.6114 | acc=0.265 F1=0.191
[SUP] ep=02 | Ltr=1.0893 Lva=1.6108 | acc=0.265 F1=0.191
[SUP] ep=03 | Ltr=0.8549 Lva=1.6113 | acc=0.294 F1=0.204
[SUP] ep=04 | Ltr=0.6879 Lva=1.5832 | acc=0.368 F1=0.320
[SUP] ep=05 | Ltr=0.5478 Lva=1.5286 | acc=0.544 F1=0.417
[SUP] ep=06 | Ltr=0.3753 Lva=1.4912 | acc=0.603 F1=0.473
[SUP] ep=07 | Ltr=0.4104 Lva=1.4918 | acc=0.662 F1=0.529
[SUP] ep=08 | Ltr=0.4125 Lva=1.4758 | acc=0.618 F1=0.474
[SUP] ep=09 | Ltr=0.3132 Lva=1.4565 | acc=0.559 F1=0.496
[SUP] ep=10 | Ltr=0.3846 Lva=1.4796 | acc=0.574 F1=0.438
[SUP] ep=11 | Ltr=0.3369 Lva=1.4530 | acc=0.662 F1=0.534
[SUP] ep=12 | Ltr=0.2784 Lva=1.4378 | acc=0.662 F1=0.535
[SUP] ep=13 | Ltr=0.2445 Lva=1.4223 | acc=0.618 F1=0.487
[SUP] ep=14 | Ltr=0.2220 Lva=1.4044 | acc=0.676 F1=0.505
[SUP] ep=15 | Ltr=0.2243 Lva=1.4171 | acc=0.662 F1=0.568
[SUP] ep=16 | Ltr=0.1530 Lva=1.3853 | acc=0.662 F1=0.499
[SUP] ep=17 | Ltr=0.2843 Lva=1.4230 | acc=0.691 F1=0.532
[SUP] ep=18 | Ltr=0.1618 Lva=1.

In [24]:
# --- Teste (com EMA em modo avaliação) + Confusion Matrix + CSV
model.eval(); ema.store(); ema.copy_to()
criterion_ce = nn.CrossEntropyLoss()

with torch.inference_mode():
    test_losses = []; logits_te = []; y_te_all = []
    for xb, yb, _ in test_loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE)
        logits = model(xb) if not USE_TTA else predict_tta(model, xb)
        test_losses.append(criterion_ce(logits, yb).item())
        logits_te.append(logits.cpu()); y_te_all.append(yb.cpu())

logits_te = torch.cat(logits_te, 0); y_te_all = torch.cat(y_te_all, 0)
ema.restore()

met_te = eval_metrics_from_logits(logits_te, y_te_all, NUM_CLASSES)
print("Teste: L={:.4f} acc={:.3f} macroF1={:.3f} ROC-AUC(OvR)={:.3f} PR-AUC(macro)={:.3f}".format(
    np.mean(test_losses), met_te["acc"], met_te["macro_f1"],
    (met_te["roc_auc_ovr"] if met_te['roc_auc_ovr']==met_te['roc_auc_ovr'] else float("nan")),
    (met_te["pr_auc_macro"] if met_te['pr_auc_macro']==met_te['pr_auc_macro'] else float("nan"))
))

y_prob = torch.softmax(logits_te, dim=1).numpy()
y_pred = y_prob.argmax(axis=1)
print("\nClassification Report (test):\n", classification_report(y_te_all.numpy(), y_pred, digits=4))

cm = confusion_matrix(y_te_all.numpy(), y_pred, labels=list(range(NUM_CLASSES)))
fig = plt.figure(figsize=(5,4))
im = plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix (test)"); plt.xlabel("Predito"); plt.ylabel("Verdadeiro")
plt.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_test_boosted.png", dpi=160); plt.close(fig)

preds_df = pd.DataFrame({"id_code": test_ids, "y_true": y_te_all.numpy(), "y_pred": y_pred})
for c in range(NUM_CLASSES): preds_df[f"p_{c}"] = y_prob[:, c]
preds_df.to_csv(OUT_DIR / "test_predictions_multiclass_boosted.csv", index=False)
print("Artefatos salvos em ./outputs/")

Teste: L=1.3566 acc=0.710 macroF1=0.553 ROC-AUC(OvR)=0.889 PR-AUC(macro)=0.642

Classification Report (test):
               precision    recall  f1-score   support

           0     0.8696    1.0000    0.9302        20
           1     0.0000    0.0000    0.0000         3
           2     0.8000    0.6667    0.7273        24
           3     0.6364    0.5385    0.5833        13
           4     0.4286    0.6667    0.5217         9

    accuracy                         0.7101        69
   macro avg     0.5469    0.5744    0.5525        69
weighted avg     0.7061    0.7101    0.7006        69

Artefatos salvos em ./outputs/
